In [1]:
!uv add datasets

Resolved 166 packages in 184ms                                       
Prepared 19 packages in 560ms                                            
Installed 19 packages in 8ms                                
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.1
 + aiosignal==1.4.0
 + async-timeout==5.0.1
 + click==8.3.0
 + datasets==4.3.0
 + dill==0.4.0
 + frozenlist==1.8.0
 + hf-xet==1.2.0
 + huggingface-hub==1.0.1
 + multidict==6.7.0
 + multiprocess==0.70.16
 + propcache==0.4.1
 + pyarrow==22.0.0
 + shellingham==1.5.4
 + tqdm==4.67.1
 + typer-slim==0.20.0
 + xxhash==3.6.0
 + yarl==1.22.0


In [2]:
from datasets import load_dataset

dataset = load_dataset("roneneldan/TinyStories")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

In [4]:
dataset = dataset["train"]['text']

In [5]:
!uv add tokenizers

Resolved 167 packages in 68ms                                        
Prepared 1 package in 40ms                                               
Installed 1 package in 3ms                                  
 + tokenizers==0.22.1


In [7]:
len(dataset)

2119719

In [9]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

# 1. Make a BPE tokenizer
tokenizer = Tokenizer(models.BPE())

# 2. Pre-tokenizer: ByteLevel is common for GPT-like BPE
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()

# 3. Decoder: ByteLevel to decode back
tokenizer.decoder = decoders.ByteLevel()

# 4. Trainer: define vocab size, min frequency, special tokens
trainer = trainers.BpeTrainer(
    vocab_size=10000,  # adjust as needed
    min_frequency=2,
    special_tokens=["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
)

# 5. Train on your list of strings
tokenizer.train_from_iterator(dataset, trainer)

# 6. Save the tokenizer if you want
tokenizer.save("bpe_tokenizer.json")

In [13]:
!uv add transformers

⠙ frozenlist==1.8.0                                                             

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Resolved 167 packages in 132ms
Prepared 4 packages in 288ms                                             
Uninstalled 1 package in 1ms
Installed 4 packages in 19ms                                
 - huggingface-hub==1.0.1
 + huggingface-hub==0.36.0
 + regex==2025.10.23
 + safetensors==0.6.2
 + transformers==4.57.1


In [15]:
from transformers import AutoTokenizer
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast(tokenizer_file="./bpe_tokenizer.json")

In [16]:
tokenizer.vocab_size

10000

In [24]:
from concurrent.futures import ThreadPoolExecutor

def encode_sentence(sentence):
    return tokenizer.encode(sentence)

with ThreadPoolExecutor() as executor:
    encoded_dataset = [
        tid
        for encoded in executor.map(encode_sentence, dataset)
        for tid in encoded
    ]


In [26]:
len(encoded_dataset)

462848638

In [29]:
import torch

# 1. Convert to PyTorch tensor (1D long tensor)
encoded_tensor = torch.tensor(encoded_dataset, dtype=torch.long)

# 2. Save it efficiently
torch.save(encoded_tensor, "encoded_dataset.pt")

print(f"Saved tensor with shape {encoded_tensor.shape} and dtype {encoded_tensor.dtype}")


Saved tensor with shape torch.Size([462848638]) and dtype torch.int64


In [27]:
import torch

class GPTChunkedDataset(torch.utils.data.Dataset):
    def __init__(self, token_ids, block_size):
        self.token_ids = token_ids
        self.block_size = block_size

        # How many full non-overlapping blocks fit?
        self.num_chunks = (len(self.token_ids) - 1) // block_size

    def __len__(self):
        return self.num_chunks

    def __getitem__(self, idx):
        start = idx * self.block_size
        x = torch.tensor(self.token_ids[start : start + self.block_size], dtype=torch.long)
        y = torch.tensor(self.token_ids[start + 1 : start + 1 + self.block_size], dtype=torch.long)
        return x, y

In [64]:
dataset = GPTChunkedDataset(encoded_dataset, block_size=128)

loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=4096,
    shuffle=True,   # Shuffle for training, optional
    drop_last=True,  # Drop partial batch
    pin_memory=True,
    num_workers=8
)

In [66]:
for xb, yb in loader:
    print(xb.shape)  # (4096, 128)     sequence_length=64
    print(yb.shape)  # (4096, 128)     sequence_length=64
    break

torch.Size([4096, 128])
torch.Size([4096, 128])


In [67]:
import torch
import torch.nn as nn

class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, vocab_size=10000, embed_dim=256, max_len=200):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.position_emb = nn.Embedding(max_len, embed_dim)

    def forward(self, x):
        batch_size, seq_len = x.size()
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        token_embedding = self.token_emb(x)
        position_embedding = self.position_emb(positions)
        return token_embedding + position_embedding           #torch.Size([256, 64, 128])    b,s,e

In [68]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_hidden_dim=None, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        
        self.attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.ln1 = nn.LayerNorm(embed_dim)

        ff_hidden_dim = ff_hidden_dim or embed_dim * 4
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_hidden_dim),
            nn.ReLU(),
            nn.Linear(ff_hidden_dim, embed_dim)
        )
        self.ln2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        x: (batch_size, seq_len, embed_dim)
        """
        batch_size, seq_len, _ = x.size()

        # Transpose for MultiheadAttention: (seq_len, batch_size, embed_dim)
        x_t = x.transpose(0, 1)

        # Make causal mask: shape (seq_len, seq_len)
        attn_mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()

        # Masked self-attention
        attn_out, _ = self.attn(x_t, x_t, x_t, attn_mask=attn_mask)

        # Residual + LayerNorm
        x2 = self.ln1(x_t + self.dropout(attn_out))

        # Feed Forward
        ff_out = self.ff(x2)

        # Residual + LayerNorm
        out = self.ln2(x2 + self.dropout(ff_out))

        # Transpose back: (batch_size, seq_len, embed_dim)
        return out.transpose(0, 1)

In [69]:
import torch
import torch.nn as nn

class MiniGPTBlock(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, num_heads=8, num_layers=4, max_len=200, dropout=0.1):
        super().__init__()
        self.embedding = TokenAndPositionEmbedding(vocab_size, embed_dim, max_len)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, dropout=dropout) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        """
        x: (batch_size, seq_len)
        returns logits: (batch_size, seq_len, vocab_size)
        """
        x = self.embedding(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits

In [92]:
model = MiniGPTBlock(vocab_size=10000)

In [93]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [94]:
device

device(type='cuda')

In [95]:
model.to(device)

MiniGPTBlock(
  (embedding): TokenAndPositionEmbedding(
    (token_emb): Embedding(10000, 256)
    (position_emb): Embedding(200, 256)
  )
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=256, bias=True)
      )
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (ln_f): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=256, out_features=10000, bias=True)
)

In [62]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()


In [96]:
!nvidia-smi

Tue Oct 28 14:31:15 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200                    On  |   00000000:83:00.0 Off |                    0 |
| N/A   38C    P0            119W /  700W |    3681MiB / 143771MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [44]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [97]:
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import OneCycleLR

epochs = 10
model.train()
scaler = GradScaler(device='cuda')
vocab_size = 10000
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, betas=(0.9, 0.98), eps=1e-9)
scheduler = OneCycleLR(optimizer, max_lr=2e-3, total_steps=epochs*len(loader))


for epoch in range(epochs):
    total_loss = 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type='cuda'):
            logits = model(batch_x)
            loss = criterion(logits.view(-1, vocab_size), batch_y.view(-1))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
    print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 3735.5455
Epoch: 2, Loss: 2254.0851
Epoch: 3, Loss: 1954.4504
Epoch: 4, Loss: 1808.9439
Epoch: 5, Loss: 1731.8213
Epoch: 6, Loss: 1685.3478
Epoch: 7, Loss: 1654.5011
Epoch: 8, Loss: 1632.7738
Epoch: 9, Loss: 1617.8856
Epoch: 10, Loss: 1610.0782


In [99]:
import torch

def generate(
    model,
    tokenizer,           # your tokenizer to decode IDs to text
    device,
    start_tokens,        # list of ints, e.g. [BOS] or your prompt encoded
    max_new_tokens=50,   # how many tokens to generate
    block_size=128,       # context window
    temperature=1.0,     # controls randomness
    top_k=None           # optional: restrict to top-k for more randomness control
):
    model.eval()
    generated = start_tokens.copy()
    input_ids = torch.tensor(start_tokens, dtype=torch.long).unsqueeze(0).to(device)  # (1, len)

    for _ in range(max_new_tokens):
        # If context is longer than block_size, crop oldest tokens
        if input_ids.size(1) > block_size:
            input_ids = input_ids[:, -block_size:]

        with torch.no_grad():
            logits = model(input_ids)  # (1, seq_len, vocab_size)

        # Get logits for last token only
        logits = logits[:, -1, :] / temperature  # (1, vocab_size)

        # Optionally top-k filter
        if top_k is not None:
            v, ix = torch.topk(logits, top_k)
            logits[logits < v[0, -1]] = -float('Inf')

        probs = torch.softmax(logits, dim=-1)  # (1, vocab_size)

        next_token = torch.multinomial(probs, num_samples=1)  # (1, 1)

        next_token_id = next_token.item()

        # Append to sequence
        generated.append(next_token_id)

        # Update input_ids
        input_ids = torch.cat([input_ids, next_token], dim=1)  # (1, seq_len+1)


    return generated

In [101]:
start_text = "Once upon a time, there was a boy named tim"
start_tokens = tokenizer.encode(start_text) # Or whatever you used

output_tokens = generate(
    model=model,
    tokenizer=tokenizer,
    device=device,
    start_tokens=start_tokens,
    max_new_tokens=250,
    block_size=128,
    temperature=0.5,
    top_k=50  # try top_k for more diverse output
)

# Decode
output_text = tokenizer.decode(output_tokens)
print(output_text)

Once upon a time, there was a boy named timmy. He was very excited because he was going to the park. He put on his shoes and ran out the door.

When he got to the park, he saw a big tree. He wanted to climb it. But he was too small to climb it. He was sad because he wanted to go up the tree.

Suddenly, he saw a little girl. She was wearing a pretty dress and had shiny shoes. She looked very pretty. The boy said, "Hi, I am Tim. Do you want to come up with me?"

The girl smiled and said, "Yes, I want to go up the tree."

Tim and the girl started to climb the tree. They climbed higher and higher. When they reached the top, they found a beautiful view. Tim said, "Thank you, girl. You are so kind."

The girl said, "You're welcome, Tim. I am happy to help."

They sat down together and enjoyed their peaceful view. Tim said, "You're welcome, little girl. We all be friends."Once upon a time, there was a little girl named Lily. She loved to play with her toys and read books. One day,


In [98]:
torch.save(model.state_dict(), 'gpt_from_scratch_H200.pth')